In [9]:
import pandas as pd
import numpy as np

In [10]:
k_values = [1, 3, 5, 7, 9, 11, 13, 15]
distance_functions = ["euclidean", "manhattan", "minkowski"]
datasets = ["bc",  "hp", "hp_imp"]

columns = ["dataset", "distance_function", "k_value", "accuracy", "precision", 
           "sensitivity", "specificity", "false_positive_rate", "f1"]
metrics_df_test = pd.DataFrame(columns=columns)
metrics_df_train = pd.DataFrame(columns=columns)


In [11]:
def get_metrics(df, type="test"):
    if type == "test":
        y_true = df["y_test"].values
        y_pred = df["y_pred"].values
    else:
        y_true = df["y_train"].values
        y_pred = df["y_pred"].values

    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))

    accuracy = (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0 
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
    false_positive_rate = FP / (FP + TN) if (FP + TN) > 0 else 0
    f1 = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0

    metrics = {
        "accuracy": accuracy,
        "precision": precision,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "false_positive_rate": false_positive_rate,
        "f1": f1
    }

    return metrics

In [12]:
for dataset in datasets:
    for dist_function in distance_functions:
        for k_value in k_values:
            df_test = pd.read_csv(f"../predictions_knn/{dataset}_predictions_k{k_value}_{dist_function}.csv")
            df_train = pd.read_csv(f"../predictions_knn/train/{dataset}_train_k{k_value}_{dist_function}.csv")
            metrics_test = get_metrics(df_test, "test")
            metrics_train = get_metrics(df_train, "train")
            other_vars = {
                "dataset": dataset,
                "k_value": k_value,
                "distance_function": dist_function
            }
            new_row_test= {**other_vars, **metrics_test}
            new_row_train = {**other_vars, **metrics_train}
            metrics_df_test.loc[len(metrics_df_test)] = new_row_test
            metrics_df_train.loc[len(metrics_df_train)] = new_row_train
            
metrics_df_test

,dataset,distance_function,k_value,accuracy,precision,sensitivity,specificity,false_positive_rate,f1
0,bc,euclidean,1,0.938053,0.925000,0.902439,0.958333,0.041667,0.913580
1,bc,euclidean,3,0.929204,0.923077,0.878049,0.958333,0.041667,0.900000
2,bc,euclidean,5,0.938053,0.947368,0.878049,0.972222,0.027778,0.911392
3,bc,euclidean,7,0.938053,0.947368,0.878049,0.972222,0.027778,0.911392
4,bc,euclidean,9,0.938053,0.947368,0.878049,0.972222,0.027778,0.911392
...,...,...,...,...,...,...,...,...,...
67,hp_imp,minkowski,7,0.806452,0.000000,0.000000,1.000000,0.000000,0.000000
68,hp_imp,minkowski,9,0.806452,0.000000,0.000000,1.000000,0.000000,0.000000
69,hp_imp,minkowski,11,0.806452,0.000000,0.000000,1.000000,0.000000,0.000000
70,hp_imp,minkowski,13,0.806452,0.000000,0.000000,1.000000,0.000000,0.000000


In [13]:
metrics_df_test.to_csv("knn_metrics.csv", index=False)
metrics_df_train.to_csv("knn_metrics_train.csv", index=False)